In [ ]:
!pip install -q streamlit streamlit-option-menu bcrypt PyJWT plotly pyngrok

In [ ]:

%%writefile /content/app.py
import os
import re
import time
import secrets
import sqlite3
import smtplib
import datetime
from contextlib import contextmanager
from email.mime.text import MIMEText

import bcrypt
import jwt
import plotly.graph_objects as go
import streamlit as st
from streamlit_option_menu import option_menu


# ============================================================
# CONFIGURATION
# ============================================================

DB_NAME = "/content/infosys_portal.db"
OTP_EXPIRY_MINUTES = 5
JWT_EXPIRY_HOURS = 2
MAX_OTP_ATTEMPTS = 5
PASSWORD_HISTORY_LIMIT = 5

JWT_SECRET = os.getenv("JWT_SECRET", "")
SENDER_EMAIL = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")

ADMIN_USERNAME = os.getenv("ADMIN_USERNAME", "")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD", "")
ADMIN_EMAIL = os.getenv("ADMIN_EMAIL", "")

if not JWT_SECRET or len(JWT_SECRET) < 32:
    st.error("Configure JWT_SECRET with a strong secret of at least 32 characters.")
    st.stop()

if not ADMIN_USERNAME or not ADMIN_PASSWORD or not ADMIN_EMAIL:
    st.error("Configure ADMIN_USERNAME, ADMIN_PASSWORD, and ADMIN_EMAIL in Colab Secrets.")
    st.stop()

os.makedirs("/content/.streamlit", exist_ok=True)

with open("/content/.streamlit/config.toml", "w") as f:
    f.write(
        '[theme]\n'
        'base="light"\n'
        'primaryColor="#B0E0E6"\n'
        'backgroundColor="#f9fcfc"\n'
        'secondaryBackgroundColor="#e3f6f5"\n'
        'textColor="#2d334a"\n'
    )

st.set_page_config(
    page_title="Infosys Portal",
    layout="wide"
)

COLORS = {
    "bg_main": "#ffffff",
    "bg_sidebar": "#fafafa",
    "text_main": "#374151",
    "text_heading": "#111827",
    "text_muted": "#6b7280",
    "accent": "#B0E0E6",
    "border": "#d1d5db",
    "border_light": "#e5e7eb"
}


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

/* Main application */
.stApp {
    background: #ffffff;
    color: #374151;
}

.main .block-container {
    max-width: 1120px;
    padding-top: 2rem;
    padding-bottom: 3rem;
}

/* Headings */
h1, h2, h3, h4 {
    color: #111827 !important;
    font-weight: 600 !important;
    letter-spacing: -0.3px;
}

/* Sidebar */
section[data-testid="stSidebar"] {
    background: #fafafa !important;
    border-right: 1px solid #e5e7eb !important;
}

/* Text inputs */
.stTextInput input,
.stNumberInput input,
.stDateInput input,
.stTextArea textarea {
    background: #ffffff !important;
    color: #202124 !important;
    border: 1px solid #d1d5db !important;
    border-radius: 6px !important;
    min-height: 42px;
    box-shadow: none !important;
}

.stTextInput input:focus,
.stNumberInput input:focus,
.stDateInput input:focus,
.stTextArea textarea:focus {
    border-color: #6b7280 !important;
    box-shadow: 0 0 0 1px #9ca3af !important;
}

/* Default buttons */
.stButton > button,
.stFormSubmitButton > button,
.stDownloadButton > button {
    background: #ffffff !important;
    background-color: #ffffff !important;
    color: #202124 !important;
    border: 1px solid #d1d5db !important;
    border-radius: 6px !important;
    min-height: 40px;
    font-weight: 500 !important;
    box-shadow: none !important;
}

/* Powder Blue primary buttons */
.stApp .stButton > button[kind="primary"],
.stApp .stFormSubmitButton > button[kind="primary"],
.stApp .stButton > button[data-testid="stBaseButton-primary"],
.stApp .stFormSubmitButton > button[data-testid="stBaseButton-primary"],
.stApp [data-testid="stFormSubmitButton"] button {
    background: #B0E0E6 !important;
    background-color: #B0E0E6 !important;
    color: #202124 !important;
    border: 1px solid #87CEEB !important;
    border-radius: 6px !important;
    min-height: 40px;
    font-weight: 600 !important;
    box-shadow: none !important;
    opacity: 1 !important;
}

/* Powder Blue hover effect */
.stApp .stButton > button[kind="primary"]:hover,
.stApp .stFormSubmitButton > button[kind="primary"]:hover,
.stApp .stButton > button[data-testid="stBaseButton-primary"]:hover,
.stApp .stFormSubmitButton > button[data-testid="stBaseButton-primary"]:hover,
.stApp [data-testid="stFormSubmitButton"] button:hover {
    background: #87CEEB !important;
    background-color: #87CEEB !important;
    color: #202124 !important;
    border-color: #6495ED !important;
}

/* Default button hover */
.stApp .stButton > button:not([kind="primary"]):hover,
.stApp .stDownloadButton > button:hover {
    background: #f3f4f6 !important;
    background-color: #f3f4f6 !important;
    border-color: #9ca3af !important;
}

/* Forms */
[data-testid="stForm"] {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 10px;
    padding: 1.25rem;
}

/* Existing custom cards */
.pn-card {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 9px;
    padding: 22px;
    box-shadow: none;
}

/* Dashboard metrics */
[data-testid="stMetric"] {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 8px;
    padding: 1rem;
}

/* Tabs */
.stTabs [data-baseweb="tab-list"] {
    gap: 1rem;
    border-bottom: 1px solid #e5e7eb;
}

/* Expanders */
[data-testid="stExpander"] {
    border: 1px solid #e5e7eb;
    border-radius: 8px;
}

/* Data tables */
[data-testid="stDataFrame"],
[data-testid="stTable"] {
    border: 1px solid #e5e7eb;
    border-radius: 8px;
    overflow: hidden;
}

/* Alerts */
[data-testid="stAlert"] {
    border-radius: 7px;
}

/* Dividers */
hr {
    border-color: #e5e7eb !important;
}

/* Responsive layout */
@media (max-width: 640px) {
    .main .block-container {
        padding: 1rem 1rem 2rem;
    }

    [data-testid="stForm"] {
        padding: 0.9rem;
    }
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# DATABASE — PRESERVE EXISTING DATABASE
# ============================================================

os.makedirs("/content", exist_ok=True)


@contextmanager
def get_db():
    conn = sqlite3.connect(DB_NAME, timeout=20)
    try:
        conn.execute("PRAGMA foreign_keys = ON")
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def hash_txt(value):
    return bcrypt.hashpw(
        value.encode("utf-8"),
        bcrypt.gensalt()
    ).decode("utf-8")


def check_txt(value, stored_hash):
    if not value or not stored_hash:
        return False

    try:
        return bcrypt.checkpw(
            value.encode("utf-8"),
            stored_hash.encode("utf-8")
        )
    except (ValueError, TypeError):
        return False


def init_db():
    with get_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE,
                email TEXT UNIQUE,
                password_hash TEXT,
                security_question TEXT,
                security_answer_hash TEXT
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS password_reset_otps (
                email TEXT PRIMARY KEY,
                otp_hash TEXT NOT NULL,
                expires_at REAL NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS password_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                password_hash TEXT NOT NULL,
                created_at REAL NOT NULL
            )
        """)


init_db()


# ============================================================
# VALIDATION
# ============================================================

def valid_email(email):
    pattern = (
        r"^[A-Za-z]{2,}[A-Za-z0-9._%+-]*"
        r"@[A-Za-z]{2,}[A-Za-z0-9-]*"
        r"\.[A-Za-z]{2,}$"
    )
    return bool(re.fullmatch(pattern, email))


def valid_password(password):
    return (
        len(password) >= 8
        and bool(re.search(r"[A-Z]", password))
        and bool(re.search(r"[a-z]", password))
        and bool(re.search(r"\d", password))
        and bool(re.search(r"[^A-Za-z0-9]", password))
    )


PASSWORD_MESSAGE = (
    "Password must contain at least 8 characters, "
    "one uppercase letter, one lowercase letter, "
    "one number, and one special character."
)


def add_password_history(email, password_hash):
    with get_db() as conn:
        conn.execute("""
            INSERT INTO password_history
            (email, password_hash, created_at)
            VALUES (?, ?, ?)
        """, (email, password_hash, time.time()))

        conn.execute("""
            DELETE FROM password_history
            WHERE email = ?
              AND id NOT IN (
                  SELECT id FROM password_history
                  WHERE email = ?
                  ORDER BY created_at DESC, id DESC
                  LIMIT ?
              )
        """, (email, email, PASSWORD_HISTORY_LIMIT))


def password_was_used(email, new_password):
    with get_db() as conn:
        rows = conn.execute("""
            SELECT password_hash
            FROM password_history
            WHERE email = ?
            ORDER BY created_at DESC, id DESC
            LIMIT ?
        """, (email, PASSWORD_HISTORY_LIMIT)).fetchall()

        current = conn.execute("""
            SELECT password_hash
            FROM users
            WHERE email = ?
        """, (email,)).fetchone()

    hashes = [row[0] for row in rows]

    if current and current[0]:
        hashes.append(current[0])

    return any(
        check_txt(new_password, old_hash)
        for old_hash in hashes
    )


# ============================================================
# JWT AUTHENTICATION
# ============================================================

def make_jwt(email, role="user", username=""):
    now = datetime.datetime.now(datetime.timezone.utc)

    payload = {
        "email": email,
        "role": role,
        "username": username,
        "iat": now,
        "exp": now + datetime.timedelta(hours=JWT_EXPIRY_HOURS)
    }

    return jwt.encode(
        payload,
        JWT_SECRET,
        algorithm="HS256"
    )


def verify_jwt(token):
    try:
        return jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )
    except (jwt.InvalidTokenError, TypeError):
        return None


# ============================================================
# EMAIL OTP
# ============================================================

def send_otp_email(email):
    if not SENDER_EMAIL or not EMAIL_PASSWORD:
        return False, (
            "Email settings are missing. Add EMAIL_ADDRESS and "
            "EMAIL_PASSWORD to Colab Secrets."
        )

    with get_db() as conn:
        user = conn.execute(
            "SELECT id FROM users WHERE email = ?",
            (email,)
        ).fetchone()

    if not user:
        return False, "No registered account exists for this email."

    otp = f"{secrets.randbelow(1000000):06d}"

    message = MIMEText(
        f"Your Infosys Portal password-reset OTP is: {otp}\n\n"
        f"This OTP expires in {OTP_EXPIRY_MINUTES} minutes.\n"
        "If you did not request this, ignore this email."
    )

    message["Subject"] = "Infosys Portal - Password Reset OTP"
    message["From"] = SENDER_EMAIL
    message["To"] = email

    try:
        with smtplib.SMTP(
            "smtp.gmail.com", 587, timeout=20
        ) as server:
            server.starttls()
            server.login(SENDER_EMAIL, EMAIL_PASSWORD)
            server.send_message(message)

        with get_db() as conn:
            conn.execute("""
                INSERT OR REPLACE INTO password_reset_otps
                (email, otp_hash, expires_at, attempts)
                VALUES (?, ?, ?, 0)
            """, (
                email,
                hash_txt(otp),
                time.time() + OTP_EXPIRY_MINUTES * 60
            ))

        return True, "OTP sent to your registered email."

    except Exception:
        return False, (
            "Could not send the email. Check your Gmail App Password "
            "and SMTP configuration."
        )


def verify_otp(email, entered_otp):
    with get_db() as conn:
        row = conn.execute("""
            SELECT otp_hash, expires_at, attempts
            FROM password_reset_otps
            WHERE email = ?
        """, (email,)).fetchone()

        if not row:
            return False, "Request an OTP first."

        otp_hash, expires_at, attempts = row

        if time.time() > expires_at:
            conn.execute(
                "DELETE FROM password_reset_otps WHERE email = ?",
                (email,)
            )
            return False, "OTP expired. Request a new OTP."

        if attempts >= MAX_OTP_ATTEMPTS:
            conn.execute(
                "DELETE FROM password_reset_otps WHERE email = ?",
                (email,)
            )
            return False, "Too many attempts. Request a new OTP."

        if not check_txt(entered_otp, otp_hash):
            conn.execute("""
                UPDATE password_reset_otps
                SET attempts = attempts + 1
                WHERE email = ?
            """, (email,))
            return False, "Incorrect OTP."

        conn.execute(
            "DELETE FROM password_reset_otps WHERE email = ?",
            (email,)
        )

    return True, "OTP verified successfully."


# ============================================================
# SESSION STATE
# ============================================================

defaults = {
    "token": None,
    "page": "Login",
    "reset_email": None,
    "reset_mode": None,
    "reset_verified": False
}

for key, value in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = value


def navigate(page):
    st.session_state.page = page
    st.rerun()


def clear_reset():
    st.session_state.reset_email = None
    st.session_state.reset_mode = None
    st.session_state.reset_verified = False


def auth_header(title, subtitle="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:24px 0;">
        <h1>Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']}">{subtitle}</p>
        <h3>{title}</h3>
    </div>
    """, unsafe_allow_html=True)


# ============================================================
# LOGIN, SIGNUP, FORGOT PASSWORD, ADMIN LOGIN
# ============================================================

if not st.session_state.token:

    if st.session_state.page not in [
        "Login", "Signup", "Forgot", "Admin Login"
    ]:
        st.session_state.page = "Login"

    _, middle, _ = st.columns([1, 1.5, 1])

    with middle:

        # ---------------- LOGIN ----------------
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")

            email = st.text_input(
                "Email address",
                key="login_email"
            ).strip().lower()

            password = st.text_input(
                "Password",
                type="password",
                key="login_password"
            )

            if st.button("Sign In →", use_container_width=True):
                if not email or not password:
                    st.error("Please enter both email and password.")

                elif not valid_email(email):
                    st.error("Enter a valid email address.")

                elif email == "infosys@ai":
                    st.error("Use Admin Login for administrator access.")

                else:
                    with get_db() as conn:
                        user = conn.execute("""
                            SELECT username, password_hash
                            FROM users
                            WHERE email = ?
                        """, (email,)).fetchone()

                    if user and check_txt(password, user[1]):
                        st.session_state.token = make_jwt(
                            email, "user", user[0]
                        )
                        navigate("Dashboard")
                    else:
                        st.error("Invalid email or password.")

            c1, c2 = st.columns(2)

            if c1.button(
                "Create Account",
                use_container_width=True,
                type="primary"
            ):
                navigate("Signup")

            if c2.button(
                "Forgot Password",
                use_container_width=True,
                type="primary"
            ):
                clear_reset()
                navigate("Forgot")

            if st.button(
                "Admin Login",
                use_container_width=True,
                type="primary"
            ):
                navigate("Admin Login")

        # ---------------- SIGNUP ----------------
        elif st.session_state.page == "Signup":
            auth_header(
                "Create an account",
                "Join Infosys Portal today"
            )

            with st.form("signup_form"):
                left, right = st.columns(2, gap="large")

                with left:
                    username = st.text_input(
                        "Full name / Username",
                        placeholder="Enter your full name"
                    )

                    email = st.text_input(
                        "Email address",
                        placeholder="Enter your email"
                    ).strip().lower()

                    password = st.text_input(
                        "Password",
                        type="password",
                        placeholder="Create a strong password"
                    )

                with right:
                    confirm_password = st.text_input(
                        "Confirm password",
                        type="password",
                        placeholder="Re-enter your password"
                    )

                    question = st.selectbox(
                        "Security Question",
                        [
                            "What is your pet name?",
                            "What is your mother's maiden name?",
                            "What is your favourite city?"
                        ]
                    )

                    answer = st.text_input(
                        "Your security answer",
                        placeholder="Enter your answer"
                    )

                st.caption(PASSWORD_MESSAGE)

                submitted = st.form_submit_button(
                    "Create Account",
                    use_container_width=True,
                    type="primary"
                )

            if submitted:
                if not all([
                    username.strip(),
                    email,
                    password,
                    confirm_password,
                    answer.strip()
                ]):
                    st.error("Please fill in all required fields.")

                elif not valid_email(email):
                    st.error("Enter a valid email address.")

                elif not valid_password(password):
                    st.error(PASSWORD_MESSAGE)

                elif password != confirm_password:
                    st.error("Passwords do not match.")

                elif email == "infosys@ai":
                    st.error("This email is reserved.")

                else:
                    password_hash = hash_txt(password)

                    try:
                        with get_db() as conn:
                            conn.execute("""
                                INSERT INTO users
                                (username, email, password_hash,
                                 security_question, security_answer_hash)
                                VALUES (?, ?, ?, ?, ?)
                            """, (
                                username.strip(),
                                email,
                                password_hash,
                                question,
                                hash_txt(answer.strip().lower())
                            ))

                        add_password_history(
                            email,
                            password_hash
                        )

                        st.success(
                            "Account created successfully. "
                            "Please sign in."
                        )

                        time.sleep(1)
                        clear_reset()
                        navigate("Login")

                    except sqlite3.IntegrityError:
                        st.error(
                            "That email or username is already registered."
                        )

            if st.button(
                "← Back to Sign In",
                use_container_width=True
            ):
                navigate("Login")

        # ---------------- FORGOT PASSWORD ----------------
        elif st.session_state.page == "Forgot":
            auth_header(
                "Reset your password",
                "Verify your identity first"
            )

            if not st.session_state.reset_email:
                email = st.text_input(
                    "Registered email address",
                    key="forgot_email"
                ).strip().lower()

                c1, c2 = st.columns(2)

                if c1.button(
                    "Security Question",
                    use_container_width=True,
                    type="primary"
                ):
                    if not email:
                        st.error("Enter your registered email.")

                    elif not valid_email(email):
                        st.error("Enter a valid email address.")

                    else:
                        with get_db() as conn:
                            user = conn.execute("""
                                SELECT security_question
                                FROM users WHERE email = ?
                            """, (email,)).fetchone()

                        if not user:
                            st.error("No account found for that email.")

                        elif not user[0]:
                            st.error(
                                "No security question is available. "
                                "Try OTP verification."
                            )

                        else:
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "sq"
                            st.session_state.reset_verified = False
                            st.rerun()

                if c2.button(
                    "Send Email OTP",
                    use_container_width=True,
                    type="primary"
                ):
                    if not email:
                        st.error("Enter your registered email.")

                    elif not valid_email(email):
                        st.error("Enter a valid email address.")

                    else:
                        ok, message = send_otp_email(email)

                        if ok:
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "otp"
                            st.session_state.reset_verified = False
                            st.success(message)
                            st.rerun()
                        else:
                            st.error(message)

            else:
                email = st.session_state.reset_email
                mode = st.session_state.reset_mode

                st.info(f"Account: {email}")

                if not st.session_state.reset_verified:

                    if mode == "sq":
                        with get_db() as conn:
                            row = conn.execute("""
                                SELECT security_question
                                FROM users WHERE email = ?
                            """, (email,)).fetchone()

                        if row:
                            st.write("Security question:")
                            st.info(row[0])

                        answer = st.text_input(
                            "Your answer",
                            key="reset_security_answer"
                        )

                        if st.button("Verify Answer"):
                            with get_db() as conn:
                                row = conn.execute("""
                                    SELECT security_answer_hash
                                    FROM users WHERE email = ?
                                """, (email,)).fetchone()

                            if row and check_txt(
                                answer.strip().lower(), row[0]
                            ):
                                st.session_state.reset_verified = True
                                st.rerun()
                            else:
                                st.error("Incorrect security answer.")

                    elif mode == "otp":
                        otp = st.text_input(
                            "Enter the 6-digit OTP",
                            max_chars=6,
                            key="reset_otp"
                        )

                        if st.button("Verify OTP"):
                            if not re.fullmatch(r"\d{6}", otp):
                                st.error("Enter the 6-digit OTP.")

                            else:
                                ok, message = verify_otp(email, otp)

                                if ok:
                                    st.session_state.reset_verified = True
                                    st.rerun()
                                else:
                                    st.error(message)

                        if st.button("Resend OTP"):
                            ok, message = send_otp_email(email)

                            if ok:
                                st.success(message)
                            else:
                                st.error(message)

                else:
                    st.success(
                        "Identity verified. Set a new password."
                    )

                    new_password = st.text_input(
                        "New password",
                        type="password",
                        key="new_password"
                    )

                    confirm_new_password = st.text_input(
                        "Confirm new password",
                        type="password",
                        key="confirm_new_password"
                    )

                    st.caption(PASSWORD_MESSAGE)

                    if st.button("Reset Password"):
                        if not new_password or not confirm_new_password:
                            st.error(
                                "Please fill in both password fields."
                            )

                        elif not valid_password(new_password):
                            st.error(PASSWORD_MESSAGE)

                        elif new_password != confirm_new_password:
                            st.error("Passwords do not match.")

                        elif password_was_used(email, new_password):
                            st.error(
                                "Choose a password you have not used recently."
                            )

                        else:
                            new_hash = hash_txt(new_password)

                            with get_db() as conn:
                                conn.execute("""
                                    UPDATE users
                                    SET password_hash = ?
                                    WHERE email = ?
                                """, (new_hash, email))

                                conn.execute("""
                                    DELETE FROM password_reset_otps
                                    WHERE email = ?
                                """, (email,))

                            add_password_history(email, new_hash)
                            clear_reset()

                            st.success(
                                "Password reset successfully. Please sign in."
                            )

                            time.sleep(1)
                            navigate("Login")

            if st.button("← Cancel / Back to Login"):
                clear_reset()
                navigate("Login")

        # ---------------- ADMIN LOGIN ----------------
        elif st.session_state.page == "Admin Login":
            auth_header(
                "Administrator Login",
                "Restricted access"
            )

            admin_username = st.text_input("Admin username")

            admin_password = st.text_input(
                "Admin password",
                type="password"
            )

            if st.button(
                "Admin Sign In",
                use_container_width=True,
                type="primary"
            ):
                if not admin_username or not admin_password:
                    st.error(
                        "Enter both administrator credentials."
                    )

                elif (
                    secrets.compare_digest(
                        admin_username, ADMIN_USERNAME
                    )
                    and secrets.compare_digest(
                        admin_password, ADMIN_PASSWORD
                    )
                ):
                    st.session_state.token = make_jwt(
                        ADMIN_EMAIL,
                        "admin",
                        ADMIN_USERNAME
                    )
                    navigate("Dashboard")

                else:
                    st.error("Invalid administrator credentials.")

            if st.button("← Back to User Login"):
                navigate("Login")


# ============================================================
# AUTHENTICATED DASHBOARDS
# ============================================================

else:
    payload = verify_jwt(st.session_state.token)

    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    role = payload.get("role")
    email = payload.get("email")
    username = payload.get("username", "")

    if role == "user":
        with get_db() as conn:
            user = conn.execute("""
                SELECT username, email
                FROM users
                WHERE email = ?
            """, (email,)).fetchone()

        if not user:
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

        username, email = user

    elif role != "admin" or email != ADMIN_EMAIL:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    with st.sidebar:
        st.markdown("## Infosys Portal")
        st.write(
            "Administrator" if role == "admin" else "User Portal"
        )

        if role == "admin":
            selected = option_menu(
                "Admin Menu",
                ["Dashboard", "Registered Users", "Logout"],
                icons=["house", "people", "box-arrow-right"]
            )

        else:
            selected = option_menu(
                "User Menu",
                ["Dashboard", "Analytics", "Reports", "Logout"],
                icons=[
                    "house",
                    "graph-up",
                    "file-text",
                    "box-arrow-right"
                ]
            )

    if selected == "Logout":
        st.session_state.token = None
        st.session_state.page = "Login"
        clear_reset()
        st.rerun()

    elif role == "admin":
        st.title("🛡️ Administrator Dashboard")
        st.write(f"Welcome, {ADMIN_USERNAME}.")

        if selected == "Registered Users":
            st.subheader("Registered Users")

            with get_db() as conn:
                users = conn.execute("""
                    SELECT username, email
                    FROM users
                    WHERE email != 'infosys@ai'
                    ORDER BY username
                """).fetchall()

            if users:
                st.dataframe(
                    [
                        {"Username": u[0], "Email": u[1]}
                        for u in users
                    ],
                    use_container_width=True,
                    hide_index=True
                )

            else:
                st.info("No registered users yet.")

        else:
            st.markdown("""
            <div class="pn-card">
                <h3>Admin Control Panel</h3>
                <p>Use Registered Users to view usernames and email addresses.</p>
            </div>
            """, unsafe_allow_html=True)

    else:
        st.title(f"Welcome, {username}!")
        st.write(f"Email: {email}")

        if selected == "Dashboard":
            st.subheader("Dashboard")

            c1, c2, c3, c4 = st.columns(4)

            for col, label, value in [
                (c1, "Documents Indexed", "128"),
                (c2, "Searches Today", "47"),
                (c3, "Efficiency Score", "98.4%"),
                (c4, "Security Status", "Secured")
            ]:
                col.markdown(f"""
                <div class="pn-card">
                    <p>{label}</p>
                    <h2>{value}</h2>
                </div>
                """, unsafe_allow_html=True)

            fig = go.Figure(
                go.Indicator(
                    mode="gauge+number",
                    value=92,
                    title={"text": "System Health Index"},
                    gauge={"axis": {"range": [0, 100]}}
                )
            )

            fig.update_layout(height=300)
            st.plotly_chart(fig, use_container_width=True)

        elif selected == "Analytics":
            st.subheader("Analytics")
            st.info("Analytics section.")

        elif selected == "Reports":
            st.subheader("Reports")
            st.info("Reports section.")


In [ ]:
import os

print("app.py exists:", os.path.exists("/content/app.py"))

In [ ]:
from google.colab import userdata

secret_names = [
    "EMAIL_ADDRESS",
    "EMAIL_PASSWORD",
    "JWT_SECRET",
    "NGROK_AUTHTOKEN",
    "ADMIN_USERNAME",
    "ADMIN_PASSWORD",
    "ADMIN_EMAIL"
]

for key in secret_names:
    try:
        value = userdata.get(key)
        print(f"{key}: {'Available' if value else 'Missing or empty'}")
    except Exception:
        print(f"{key}: Missing or inaccessible")

In [ ]:

import os
import time
import socket
import subprocess
from google.colab import userdata
from pyngrok import ngrok

# ============================================================
# 1. LOAD GOOGLE COLAB SECRETS
# ============================================================

secret_names = [
    "JWT_SECRET",
    "NGROK_AUTHTOKEN",
    "EMAIL_ADDRESS",
    "EMAIL_PASSWORD",
    "ADMIN_USERNAME",
    "ADMIN_PASSWORD",
    "ADMIN_EMAIL"
]

for key in secret_names:
    try:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
    except Exception:
        pass

# Check required secrets
required = [
    "JWT_SECRET",
    "NGROK_AUTHTOKEN",
    "ADMIN_USERNAME",
    "ADMIN_PASSWORD",
    "ADMIN_EMAIL"
]

missing = [
    key for key in required
    if not os.environ.get(key)
]

if missing:
    raise ValueError(
        "Missing Colab Secrets: " + ", ".join(missing)
    )

# ============================================================
# 2. CONFIGURATION
# ============================================================

PORT = 8501
APP_FILE = "/content/app.py"
LOG_FILE = "/content/streamlit.log"

if not os.path.exists(APP_FILE):
    raise FileNotFoundError(
        "app.py not found. Run your app.py code cell first."
    )

# ============================================================
# 3. CLEAN UP PREVIOUS STREAMLIT PROCESS
# ============================================================

print("Cleaning up previous Streamlit processes...")

# Stop the process occupying port 8501, if any.
subprocess.run(
    ["bash", "-lc", f"fuser -k {PORT}/tcp"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

# Close previous ngrok tunnels.
ngrok.kill()
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

# ============================================================
# 4. START STREAMLIT
# ============================================================

log_file = open(LOG_FILE, "w")
process = None

try:
    process = subprocess.Popen(
        [
            "streamlit",
            "run",
            APP_FILE,
            "--server.port", str(PORT),
            "--server.address", "0.0.0.0",
            "--server.headless", "true",
            "--browser.gatherUsageStats", "false"
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

    # Wait until Streamlit is ready.
    started = False

    for _ in range(30):
        if process.poll() is not None:
            break

        try:
            with socket.create_connection(
                ("127.0.0.1", PORT),
                timeout=1
            ):
                started = True
                break
        except OSError:
            time.sleep(1)

    log_file.flush()

    if not started:
        print("Streamlit failed to start.")
        with open(LOG_FILE, "r") as f:
            print(f.read())

        raise RuntimeError(
            "Check the Streamlit log above."
        )

    # ========================================================
    # 5. CREATE NGROK PUBLIC URL
    # ========================================================

    tunnel = ngrok.connect(PORT, "http")

    print("\n" + "=" * 55)
    print("ACCOUNT PORTAL IS RUNNING")
    print("=" * 55)
    print("Portal URL:", tunnel.public_url)
    print("=" * 55)
    print("Open the URL in your browser.")
    print("Keep this cell running while using the portal.")
    print("To stop: press Ctrl+C or click the Colab Stop button.")
    print("=" * 55)

    # Keep the launcher active while Streamlit is running.
    while process.poll() is None:
        time.sleep(2)

    log_file.flush()
    print("\nStreamlit has stopped.")

    with open(LOG_FILE, "r") as f:
        print(f.read()[-3000:])

except KeyboardInterrupt:
    print("\nStopping Account Portal...")

except Exception as e:
    print("Launcher error:", e)

finally:
    # ========================================================
    # 6. SHUT DOWN THE APPLICATION
    # ========================================================

    if process is not None and process.poll() is None:
        process.terminate()

        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()

    try:
        ngrok.kill()
    except Exception:
        pass

    log_file.close()

    print("Portal launcher stopped.")
    print("Run this cell again to restart the portal.")
